# Phase 1C — TrackNetV3 shuttle tracking on a Colab GPU

Tracks the shuttle in every clip from Phase 1A and writes one `<clip>_ball.csv` (`Frame, Visibility, X, Y` in the source video's 1280×720 pixels) per clip.

**Before running**
1. On your Mac: `.venv/bin/python shuttle_track.py pack`, then upload `data/colab/tracknet_input_512.zip` to a Google Drive folder named **`badminton-tracknet`** (in My Drive). An older full-size `tracknet_input.zip` works too; the 512×288 one is about 5× faster.
2. Here: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Each clip's result is saved to Drive as soon as it finishes, so if the session disconnects, just run all again: finished clips are skipped.

To track more matches later, pack just those (`pack --match <id>`, repeatable) and upload the new zip over the old one; the notebook unpacks it afresh and keeps every earlier result.

**After it finishes**: download `badminton-tracknet/tracknet_output.zip`, then on your Mac run `.venv/bin/python shuttle_track.py ingest ~/Downloads/tracknet_output.zip`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/badminton-tracknet"
INPUT_ZIPS = ["tracknet_input_512.zip", "tracknet_input.zip"]  # the first one found in DRIVE_DIR is used
OUTPUT_DIR = f"{DRIVE_DIR}/output"   # one CSV per clip, written as each finishes
BATCH_SIZE = 16
LIMIT = None                         # set to e.g. 3 for a quick trial run

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

## Set up TrackNetV3

Pinned to the commit this pipeline was tested against. Its `requirements.txt` pins torch 1.10 and numpy 1.22, which don't install on current Colab, so it's skipped: Colab's preinstalled PyTorch, NumPy, OpenCV and pandas work. Only the missing packages are added (`pycocotools` is imported by `test.py` but not listed).

In [ ]:
%%bash
set -e
cd /content
if [ ! -d TrackNetV3 ]; then
  git clone -q https://github.com/qaz812345/TrackNetV3.git
  git -C TrackNetV3 checkout -q 6eda442
fi
pip install -q parse pycocotools gdown
cd TrackNetV3
if [ ! -f ckpts/TrackNet_best.pt ]; then
  gdown -q 1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA -O TrackNetV3_ckpts.zip
  unzip -q -o TrackNetV3_ckpts.zip && rm TrackNetV3_ckpts.zip
fi
ls -la ckpts

In [ ]:
import json, os, shutil, zipfile

INPUT_ZIP = next((f"{DRIVE_DIR}/{z}" for z in INPUT_ZIPS if os.path.exists(f"{DRIVE_DIR}/{z}")), None)
assert INPUT_ZIP, f"Upload {INPUT_ZIPS[0]} to {DRIVE_DIR}"
LOCAL_IN = "/content/" + os.path.basename(INPUT_ZIP)[:-4]

# Copy the zip once from Drive (one big sequential read) and unpack it on the local disk. A different zip
# uploaded under the same name (other matches, say) replaces whatever an earlier run unpacked.
SOURCE = f"{os.path.getsize(INPUT_ZIP)} {os.path.getmtime(INPUT_ZIP)}"
STAMP = f"{LOCAL_IN}/.source"
if not os.path.exists(STAMP) or open(STAMP).read() != SOURCE:
    shutil.rmtree(LOCAL_IN, ignore_errors=True)
    shutil.copy(INPUT_ZIP, "/content/input.zip")
    with zipfile.ZipFile("/content/input.zip") as z:
        z.extractall(LOCAL_IN)
    os.remove("/content/input.zip")
    with open(STAMP, "w") as f:
        f.write(SOURCE)

matches = sorted(d for d in os.listdir(LOCAL_IN) if os.path.isdir(f"{LOCAL_IN}/{d}"))
clips = {m: sorted(f for f in os.listdir(f"{LOCAL_IN}/{m}") if f.endswith(".mp4")) for m in matches}


def scale_of(match):
    """Clips may be shrunk to TrackNetV3's input size; results are scaled back to the source video."""
    path = f"{LOCAL_IN}/{match}/pack.json"
    if not os.path.exists(path):
        return 1.0, 1.0
    info = json.load(open(path))
    return info["source_size"][0] / info["clip_size"][0], info["source_size"][1] / info["clip_size"][1]


print(os.path.basename(INPUT_ZIP), {m: len(c) for m, c in clips.items()}, "| total", sum(map(len, clips.values())))

## Track every clip

Runs TrackNetV3's own `predict.py` (TrackNet + InpaintNet, temporal-ensemble mode, streaming frames with `--large_video`) with two changes, neither of which touches its code:

- **Background image.** `predict.py` builds it per clip from up to 1,800 full-size frames held in memory, which can exhaust Colab's RAM on long clips. Phase 1B already made a clean per-match median, `court_reference.jpg`, and a per-match median is also what TrackNetV3 was trained with. On a test clip both gave identical positions.
- **In-process data loading.** `predict.py` starts as many worker processes as the batch size (16) for every clip; InpaintNet's coordinate data doesn't need them.

The clips in `tracknet_input_512.zip` are already at TrackNetV3's 512×288 input size, which saves it resizing every frame eight times over on one CPU core. Each result is scaled back to the source video's pixels (from the match's `pack.json`) before it's saved.

The first clip is slower while PyTorch warms up. With `LIMIT = 3`, the printed minutes tell you roughly how long all 445 clips will take.

In [ ]:
import contextlib, io, runpy, sys, time
import cv2, numpy as np, pandas as pd
import torch.utils.data as tud
from PIL import Image

os.chdir("/content/TrackNetV3")
sys.path.insert(0, "/content/TrackNetV3")
import dataset
from utils.general import WIDTH, HEIGHT

BG_MODE = torch.load("ckpts/TrackNet_best.pt", map_location="cpu", weights_only=False)["param_dict"]["bg_mode"]


class InProcessLoader(tud.DataLoader):
    """predict.py starts as many worker processes as the batch size; InpaintNet's coordinate data
    doesn't need them, and on a 2-CPU Colab machine they're pure overhead for every clip."""
    def __init__(self, *args, **kwargs):
        kwargs["num_workers"] = 0
        super().__init__(*args, **kwargs)


tud.DataLoader = InProcessLoader  # predict.py's `from torch.utils.data import DataLoader` picks this up


def match_median(match):
    """The per-match median in the form Video_IterableDataset.__gen_median__ returns."""
    rgb = np.ascontiguousarray(cv2.imread(f"{LOCAL_IN}/{match}/court_reference.jpg")[..., ::-1])
    if BG_MODE == "concat":
        return np.moveaxis(np.array(Image.fromarray(rgb).resize((WIDTH, HEIGHT))), -1, 0)
    return rgb.astype(float)


def track_clip(video, save_dir):
    sys.argv = ["predict.py", "--video_file", video, "--tracknet_file", "ckpts/TrackNet_best.pt",
                "--inpaintnet_file", "ckpts/InpaintNet_best.pt", "--save_dir", save_dir,
                "--large_video", "--batch_size", str(BATCH_SIZE)]
    log = io.StringIO()
    try:
        with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            runpy.run_path("predict.py", run_name="__main__")
    except Exception:
        print(log.getvalue()[-2000:])
        raise


def save_scaled(src, dst, scale):
    """Write predict.py's CSV with X, Y in source-video pixels (undetected frames stay 0, 0)."""
    df = pd.read_csv(src)
    df["X"] = (df["X"] * scale[0]).round(1)
    df["Y"] = (df["Y"] * scale[1]).round(1)
    df.to_csv(dst, index=False)
    os.remove(src)


todo = [(m, c) for m in matches for c in clips[m]][:LIMIT]
done = failed = 0
t0 = time.time()
current = None
for m, c in todo:
    out_dir = f"{OUTPUT_DIR}/{m}"
    target = f"{out_dir}/{c[:-4]}_ball.csv"
    done += 1
    if os.path.exists(target):
        continue
    if m != current:
        median = match_median(m)
        dataset.Video_IterableDataset.__gen_median__ = lambda self, *args, _m=median: _m
        scale = scale_of(m)
        current = m
        os.makedirs(out_dir, exist_ok=True)
    try:
        track_clip(f"{LOCAL_IN}/{m}/{c}", "/content/pred")
        save_scaled(f"/content/pred/{c[:-4]}_ball.csv", target, scale)  # lands on Drive only when complete
    except Exception as e:
        failed += 1
        print(f"FAILED {m}/{c}: {e!r}")
    print(f"{done}/{len(todo)}  {m}/{c}  {(time.time() - t0) / 60:.1f} min", flush=True)
print(f"finished, {failed} failed")

In [ ]:
n = sum(os.path.exists(f"{OUTPUT_DIR}/{m}/{c[:-4]}_ball.csv") for m in matches for c in clips[m])
print(f"{n} of the {sum(map(len, clips.values()))} clips in {os.path.basename(INPUT_ZIP)} tracked")
shutil.make_archive(f"{DRIVE_DIR}/tracknet_output", "zip", OUTPUT_DIR)  # every match tracked so far
print(f"Download {DRIVE_DIR}/tracknet_output.zip, then on your Mac:")
print("  .venv/bin/python shuttle_track.py ingest ~/Downloads/tracknet_output.zip")